# My own signs: teaching a model NAME and MY

The 250-sign model on the website doesn't know **NAME**, and it isn't great at **MY**. So this notebook trains a small model on my own recordings. It reads one whole sign (both hands and the body) and says which of my words it was, or **other**.

On the website, both models read every sign. When this one is at least 80% sure it saw one of its words, its answer wins; otherwise the 250-sign model's answer is used.

Same plan as the J and Z notebook (`train_moves.ipynb`): load → turn into numbers → train → test → save.

## 1. Upload your recordings

Click the **folder icon** on the left and upload every `nm-signs-….csv` file from `training/data`. If you've recorded an exam (`nm-signs-exam.csv`: a different day, or a friend), upload that too.

In [ ]:
import glob, json
import numpy as np
import pandas as pd
import tensorflow as tf

practice_files = sorted(glob.glob("nm-signs-*.csv"), key=lambda name: int(name.split("-")[-1].split(".")[0]))
EXAM_FILE = "nm-signs-exam.csv"

print(len(practice_files), "practice files:", practice_files)
print("Exam file uploaded:", EXAM_FILE in glob.glob("*.csv"))

## 2. Load the clips

In Signs mode the app cuts the camera stream into signs for you: raise your hands → sign → lower them. Each cut sign is a **clip**, and the `clip` column says which rows belong together. Every row is one frame: x, y, z for 75 points (left hand 21, right hand 21, body 33). An empty cell means that point wasn't seen.

The download holds everything since the page was opened, so the same clip can be in two files; repeats are removed.

In [ ]:
def clips_in(file):
    """One recording → a list of (label, clip). A clip is frames × 75 points × (x, y, z); NaN = not seen."""
    table = pd.read_csv(file)
    found = []
    for _, rows in table.groupby("clip", sort=True):
        clip = rows.drop(columns=["label", "clip"]).values.reshape(len(rows), 75, 3)
        label = rows["label"].iloc[0]
        found.append(("other" if label.lower() == "other" else label, clip))  # "OTHER" typed in the Word box = other
    return found

def without_repeats(pairs):
    """The app's download holds everything since the page was opened, so the same clip can be in two files."""
    seen, kept = set(), []
    for label, clip in pairs:
        key = np.nan_to_num(clip).round(4).tobytes()
        if key not in seen:
            seen.add(key)
            kept.append((label, clip))
    return kept

practice = without_repeats([pair for file in practice_files for pair in clips_in(file)])
print(pd.Series([label for label, _ in practice]).value_counts())

## 3. Drop the twitches

A clip with fewer than 10 frames that have a hand in them was a twitch, not a sign.

In [ ]:
MIN_HAND_FRAMES = 10  # fewer frames with a hand in them = a twitch, not a sign

def hand_frames(clip):
    return int((~np.isnan(clip[:, 0, 0]) | ~np.isnan(clip[:, 21, 0])).sum())

before = len(practice)
practice = [(label, clip) for label, clip in practice if hand_frames(clip) >= MIN_HAND_FRAMES]
print("Dropped", before - len(practice), "clips that were too short. Left:")
print(pd.Series([label for label, _ in practice]).value_counts())

## 4. Turn each clip into numbers

Like the J and Z moves, every clip becomes the same size: **24 frames**. For each frame we keep:

1. **Measured from the middle of your shoulders, in shoulder widths**, so it doesn't matter where you stand or how close you are.
2. **The signing hand first**: the hand seen in more frames (the same rule as the 250-sign model). Left-handed signing is mirrored so it looks right-handed.
3. **47 points:** the signing hand (21), the other hand (21), both wrists and elbows (4), and your nose (1), with x and y for each.
4. **2 flags** saying whether each hand is seen. A missing hand keeps its last position.

That's **47 × 2 + 2 = 96 numbers per frame**. The website does exactly the same in `src/lib/asl/signModel.ts`, and a test there checks that both give the same numbers.

In [ ]:
FRAMES = 24
# Where things are inside a clip frame: left hand 0–20, right hand 21–41, body 42–74.
LEFT_HAND, RIGHT_HAND, BODY = 0, 21, 42
NOSE = BODY + 0
SHOULDER = {"left": BODY + 11, "right": BODY + 12}
ELBOW = {"left": BODY + 13, "right": BODY + 14}
WRIST = {"left": BODY + 15, "right": BODY + 16}

def fill_gaps(track):
    """A point's missing frames copy the last frame it was seen in (or the first, at the start). Never seen → 0."""
    track = track.copy()
    seen = np.where(~np.isnan(track[:, 0]))[0]
    if not len(seen):
        return np.zeros_like(track)
    track[: seen[0]] = track[seen[0]]
    for t in range(seen[0] + 1, len(track)):
        if np.isnan(track[t, 0]):
            track[t] = track[t - 1]
    return track

def resample(frames, count=FRAMES):
    """Stretches or squashes a clip to exactly `count` frames, blending neighbouring frames."""
    t = np.linspace(0, len(frames) - 1, count)
    before = np.floor(t).astype(int)
    after = np.minimum(before + 1, len(frames) - 1)
    mix = (t - before)[:, None]
    return frames[before] * (1 - mix) + frames[after] * mix

def sign_features(clip):
    """The same recipe as signFeatures() in src/lib/asl/signModel.ts."""
    xy = np.asarray(clip, dtype=np.float64)[:, :, :2]
    # 1. Measure everything from the middle of the shoulders, in shoulder widths.
    both = ~np.isnan(xy[:, SHOULDER["left"], 0]) & ~np.isnan(xy[:, SHOULDER["right"], 0])
    left, right = xy[both, SHOULDER["left"]], xy[both, SHOULDER["right"]]
    centre, width = np.array([0.5, 0.6]), 0.3  # a guess, for when the shoulders aren't seen
    if both.any() and np.linalg.norm(left - right, axis=1).mean() > 0:
        centre = ((left + right) / 2).mean(axis=0)
        width = np.linalg.norm(left - right, axis=1).mean()
    xy = (xy - centre) / width
    # 2. The hand seen in more frames is the signing hand (the same rule as the 250-word model).
    #    Mirror left-handed signing so it looks right-handed.
    seen_frames = {name: int((~np.isnan(xy[:, start, 0])).sum()) for name, start in [("left", LEFT_HAND), ("right", RIGHT_HAND)]}
    main = "right" if seen_frames["right"] >= seen_frames["left"] else "left"
    other = "left" if main == "right" else "right"
    if main == "left":
        xy[..., 0] *= -1
    hand = {"left": LEFT_HAND, "right": RIGHT_HAND}
    # 3. Per frame: signing hand (21 dots), other hand (21), wrists and elbows, nose, and whether each hand is seen.
    parts = [xy[:, hand[main] + i] for i in range(21)] + [xy[:, hand[other] + i] for i in range(21)]
    parts += [xy[:, WRIST[main]], xy[:, ELBOW[main]], xy[:, WRIST[other]], xy[:, ELBOW[other]], xy[:, NOSE]]
    points = np.stack([fill_gaps(track) for track in parts], axis=1).reshape(len(xy), -1)
    seen = np.stack([~np.isnan(xy[:, hand[main], 0]), ~np.isnan(xy[:, hand[other], 0])], axis=1)
    return resample(np.concatenate([points, seen], axis=1)).astype(np.float32)

FEATURES = 47 * 2 + 2  # 47 points × (x, y) + 2 "hand seen" flags = 96
print("One clip becomes:", sign_features(practice[0][1]).shape)

## 5. Practice pile and test pile

If you uploaded an exam, that's the test. If not, **every 5th clip of each word is kept aside as a quiz**. A quiz from the same session is easier than a real exam, so expect a lower score on a new day or with a new person.

In [ ]:
words = sorted({label for label, _ in practice} - {"other"})
LABELS = words + ["other"]
print("The model will learn:", LABELS)

if EXAM_FILE in glob.glob("*.csv"):
    test = [(label, clip) for label, clip in without_repeats(clips_in(EXAM_FILE)) if hand_frames(clip) >= MIN_HAND_FRAMES]
    train = practice
    print("Exam: your exam recording,", len(test), "clips the model never studies.")
else:
    # No exam recording yet: keep every 5th clip of each word aside as a quiz.
    # (A quiz from the same session is easier than a real exam, so expect a lower score on new days or people.)
    test, train, count = [], [], {}
    for label, clip in practice:
        count[label] = count.get(label, 0) + 1
        (test if count[label] % 5 == 0 else train).append((label, clip))
    print("No exam file, so", len(test), "clips are kept aside as a quiz.")

X_train = np.stack([sign_features(clip) for _, clip in train])
y_train_raw = np.array([LABELS.index(label) for label, _ in train])
X_test = np.stack([sign_features(clip) for _, clip in test])
y_test = np.array([LABELS.index(label) for label, _ in test])
print("Train:", X_train.shape, " Test:", X_test.shape)

## 6. Make extra practice copies

The same idea as before: 10 slightly changed copies of every clip (tilted, resized, standing a bit to the side, faster or slower, a bit shaky), because NAME is still NAME when you do it slightly differently.

In [ ]:
rng = np.random.default_rng(42)

def augment(features):
    """A slightly changed copy of one clip: tilted, resized, shifted, sped up or slowed down, a bit shaky."""
    f = features.copy()
    points = f[:, :94].reshape(FRAMES, 47, 2)
    angle = np.radians(rng.uniform(-10, 10))  # tilt by up to 10 degrees
    c, s = np.cos(angle), np.sin(angle)
    x, y_ = points[..., 0].copy(), points[..., 1].copy()
    points[..., 0], points[..., 1] = c * x - s * y_, s * x + c * y_
    points *= rng.uniform(0.9, 1.1)  # up to 10% bigger or smaller
    points += rng.uniform(-0.1, 0.1, size=2)  # stand a little to the side
    f[:, :94] = points.reshape(FRAMES, 94)
    t = np.linspace(0, 1, FRAMES) ** rng.uniform(0.75, 1.33) * (FRAMES - 1)  # faster at the start or the end
    before = np.floor(t).astype(int)
    after = np.minimum(before + 1, FRAMES - 1)
    mix = (t - before)[:, None]
    f = f[before] * (1 - mix) + f[after] * mix
    f[:, :94] += rng.normal(0, 0.02, (FRAMES, 94))  # a tiny bit of shake
    return f.astype(np.float32)

X_aug = np.concatenate([X_train] + [np.stack([augment(f) for f in X_train]) for _ in range(10)])
y_aug = np.concatenate([y_train_raw] * 11)
print("Training examples:", len(X_aug), "(your", len(X_train), "clips + 10 changed copies of each)")

## 7. Build the model

The same shape as the J and Z model: two **Conv1D** layers slide along the 24 frames looking for patterns ("two hands meet, then tap twice"), then a final layer gives one score per word plus "other". The only difference is the input: 96 numbers per frame instead of 63.

In [ ]:
def make_model(batch_size=None):
    return tf.keras.Sequential([
        tf.keras.Input(shape=(FRAMES, FEATURES), batch_size=batch_size),
        tf.keras.layers.Conv1D(32, 5, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling1D(2),
        tf.keras.layers.Conv1D(32, 3, activation="relu", padding="same"),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(len(LABELS), activation="softmax"),
    ])

tf.keras.utils.set_random_seed(42)  # same starting dials every run, so you get the same result
model = make_model()
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 8. Train

In [ ]:
history = model.fit(X_aug, y_aug, epochs=30, batch_size=32, verbose=2)

## 9. Test

Clips the model never studied. Rows = what the sign really was, columns = what the model said. Everything on the diagonal is right.

In [ ]:
scores = model.predict(X_test, verbose=0)
guess = scores.argmax(axis=1)
print(f"Score: {(guess == y_test).mean():.0%} of {len(y_test)} clips the model never studied\n")
print(pd.crosstab(pd.Series([LABELS[i] for i in y_test], name="really"),
                  pd.Series([LABELS[i] for i in guess], name="model said")))

## 10. How sure does it need to be?

The website only uses this model's answer when it's at least 80% sure. Otherwise the 250-sign model decides.

In [ ]:
SURE = 0.8
sure = scores.max(axis=1) >= SURE
word = guess != LABELS.index("other")
print(f"The model was at least {SURE:.0%} sure on {sure.mean():.0%} of the test clips,")
print(f"and right on {(guess[sure] == y_test[sure]).mean():.0%} of those.")
print("Times the website would have written a wrong word:", int((sure & word & (guess != y_test)).sum()))

## 11. Save the model for the website

A fixed shape again: exactly one clip, `[1, 24, 96]`.

In [ ]:
one_at_a_time = make_model(batch_size=1)  # the website needs a fixed shape: exactly 1 clip
one_at_a_time.set_weights(model.get_weights())
converter = tf.lite.TFLiteConverter.from_keras_model(one_at_a_time)
with open("my_signs.tflite", "wb") as f:
    f.write(converter.convert())
with open("my_signs.json", "w") as f:
    json.dump(LABELS, f)

# Check: the saved file gives the same answers as the model here.
interpreter = tf.lite.Interpreter(model_path="my_signs.tflite")
interpreter.allocate_tensors()
inp, out = interpreter.get_input_details()[0], interpreter.get_output_details()[0]
same = 0
for features, expected in zip(X_test, scores):
    interpreter.set_tensor(inp["index"], features[None])
    interpreter.invoke()
    same += np.allclose(interpreter.get_tensor(out["index"])[0], expected, atol=1e-4)
print("Input shape:", inp["shape"], " Saved model agrees on", same, "of", len(X_test), "test clips")

## 12. Download

This downloads `my_signs.tflite` (the model) and `my_signs.json` (the words). They go in your project at `public/models/my_signs/`.

In [ ]:
from google.colab import files
files.download("my_signs.tflite")
files.download("my_signs.json")